# 1: Bronze to Silver Airbnb and Official Licenses

This notebook implements the first stage of the Medallion Architecture.

The final project uses Airbnb as the main dataset and official tourist license records as an external regulatory source.

Inputs:
- data/bronze/airbnb_raw.csv
- data/bronze/licenses_raw.csv

Outputs:
- data/silver/airbnb_clean.parquet
- data/silver/licenses_clean.parquet
- data/silver/airbnb_license_reference.parquet
- reports/tables/silver_data_quality_summary.csv

This notebook does not perform model training, target construction, geospatial matching, SHAP, deep learning or scoring. Its purpose is to create clean, traceable Silver datasets.

In [102]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

In [103]:
def find_project_root(start_path=None, project_folder_name="datasets"):
    """
    Find the project root by walking upwards from the current working directory.
    This avoids path errors when the notebook is executed from a subfolder.
    """
    current_path = Path.cwd() if start_path is None else Path(start_path).resolve()

    for path in [current_path] + list(current_path.parents):
        if path.name == project_folder_name:
            return path

    raise RuntimeError(
        f"Could not find project root folder named '{project_folder_name}'. "
        f"Current working directory is: {Path.cwd()}"
    )


PROJECT_DIR = find_project_root()

DATA_DIR = PROJECT_DIR / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

REPORTS_DIR = PROJECT_DIR  / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_DIR / "models"

AIRBNB_RAW_PATH = BRONZE_DIR / "airbnb_raw.csv"
LICENSES_RAW_PATH = BRONZE_DIR / "licenses_raw.csv"

AIRBNB_SILVER_PATH = SILVER_DIR / "airbnb_clean.parquet"
LICENSES_SILVER_PATH = SILVER_DIR / "licenses_clean.parquet"
AIRBNB_LICENSE_REFERENCE_PATH = SILVER_DIR / "airbnb_license_reference.parquet"

SILVER_QUALITY_SUMMARY_PATH = TABLES_DIR / "silver_data_quality_summary.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("AIRBNB_RAW_PATH:", AIRBNB_RAW_PATH)
print("LICENSES_RAW_PATH:", LICENSES_RAW_PATH)

PROJECT_DIR: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets
AIRBNB_RAW_PATH: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\data\bronze\airbnb_raw.csv
LICENSES_RAW_PATH: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\data\bronze\licenses_raw.csv


In [104]:
required_files = {
    "Airbnb raw": AIRBNB_RAW_PATH,
    "Licenses raw": LICENSES_RAW_PATH,
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at: {path}")

print("All Bronze input files found.")

All Bronze input files found.


In [ ]:
# Data layers
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Report outputs
REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

# Model outputs
MODELS_DIR = PROJECT_DIR / "models"

directories_to_create = [
    SILVER_DIR,
    GOLD_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    MODELS_DIR,
]

for directory in directories_to_create:
    directory.mkdir(parents=True, exist_ok=True)

print("Created / verified project directories:")
for directory in directories_to_create:
    print(f"- {directory}")

In [105]:
airbnb_raw = pd.read_csv(AIRBNB_RAW_PATH)
licenses_raw = pd.read_csv(LICENSES_RAW_PATH)

print("Airbnb raw shape:", airbnb_raw.shape)
print("Licenses raw shape:", licenses_raw.shape)

Airbnb raw shape: (19410, 18)
Licenses raw shape: (10726, 21)


In [106]:
print("Airbnb raw columns:")
display(pd.DataFrame({"column": airbnb_raw.columns}))

print("Licenses raw columns:")
display(pd.DataFrame({"column": licenses_raw.columns}))

Airbnb raw columns:


,column
0,id
1,name
2,host_id
3,host_name
4,neighbourhood_group
5,neighbourhood
6,latitude
7,longitude
8,room_type
9,price


Licenses raw columns:


,column
0,N_EXPEDIENT
1,CODI_DISTRICTE
2,NOM_DISTRICTE
3,CODI_BARRI
4,NOM_BARRI
5,TIPUS_CARRER
6,CARRER
7,TIPUS_NUM
8,NUM1
9,LLETRA1


## Silver columns:
 - Here we select the columns we need for the silver data, since we do not want to lose potentially useful data yet, nan values will be handled later. However we do need to anomize so we create a listing key

In [107]:
AIRBNB_SILVER_COLUMNS = [
    # Anonymous technical identifier for internal traceability
    "listing_key",

    # Text
    "text",

    # Location
    "neighbourhood_group",
    "neighbourhood",
    "municipality",
    "latitude",
    "longitude",
    "has_valid_coordinates",

    # Listing characteristics
    "property_type",
    "minimum_nights",
    "availability_365",

    # Economic variables
    "price",
    "log_price",
    "has_invalid_price",

    # Review / activity variables
    "number_of_reviews",
    "last_review",
    "reviews_per_month",
    "number_of_reviews_ltm",
    "calculated_host_listings_count",

    # Objective textual features
    "text_length",
    "word_count",
    "uppercase_ratio",
    "digit_count",
    "exclamation_count",
    "empty_text",
]

## Helper functions: 
This functions will be the ones used during the preprocessing steps

In [108]:
def clean_text_value(value):
    """
    Convert a value to a clean string.
    Missing values are converted to an empty string.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def safe_numeric(series):
    """
    Convert a pandas Series to numeric values.
    Invalid parsing is converted to NaN.
    """
    return pd.to_numeric(series, errors="coerce")


def compute_log_price(price_series):
    """
    Compute log(price) only for strictly positive prices.
    Non-positive or missing values are converted to NaN.
    """
    price = safe_numeric(price_series)
    return np.where(price > 0, np.log(price), np.nan)
    

In [109]:
def compute_uppercase_ratio(text):
    """
    Compute the ratio of uppercase alphabetic characters over total alphabetic characters.
    """
    text = clean_text_value(text)
    letters = [char for char in text if char.isalpha()]

    if len(letters) == 0:
        return 0.0

    uppercase_letters = [char for char in letters if char.isupper()]
    return len(uppercase_letters) / len(letters)


def compute_text_features(text_series):
    """
    Compute objective text-based features from a text Series.

    These features describe length and formatting patterns.
    They do not rely on manually selected suspicious keywords.
    """
    text = text_series.fillna("").astype(str)

    features = pd.DataFrame(index=text.index)
    features["text_length"] = text.str.len()
    features["word_count"] = text.str.split().str.len()
    features["uppercase_ratio"] = text.apply(compute_uppercase_ratio)
    features["digit_count"] = text.apply(lambda value: sum(char.isdigit() for char in value))
    features["exclamation_count"] = text.str.count("!")

    return features

## Clean Airbnb Dataset:
Now we clean the airbnb dataset, we will obtain the silver features mentioned before

In [110]:
airbnb_clean = pd.DataFrame()

# Anonymous technical key for internal traceability
airbnb_clean["listing_key"] = [
    f"airbnb_{idx:06d}" for idx in range(len(airbnb_raw))
]

# Text field
airbnb_clean["text"] = (
    airbnb_raw["name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Location fields
airbnb_clean["neighbourhood_group"] = airbnb_raw.get("neighbourhood_group", np.nan)
airbnb_clean["neighbourhood"] = airbnb_raw.get("neighbourhood", np.nan)

# In this dataset, municipality is approximated using neighbourhood.
# This creates a consistent geographic column for later EDA/modeling.
airbnb_clean["municipality"] = (
    airbnb_raw["neighbourhood"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
)

# Listing type
airbnb_clean["property_type"] = (
    airbnb_raw["room_type"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
)

# Economic fields
airbnb_clean["price"] = safe_numeric(airbnb_raw["price"])
airbnb_clean["log_price"] = compute_log_price(airbnb_clean["price"])

# Listing/activity fields
airbnb_clean["minimum_nights"] = safe_numeric(
    airbnb_raw.get("minimum_nights", np.nan)
)

airbnb_clean["number_of_reviews"] = safe_numeric(
    airbnb_raw.get("number_of_reviews", np.nan)
)

airbnb_clean["reviews_per_month"] = safe_numeric(
    airbnb_raw.get("reviews_per_month", np.nan)
)

airbnb_clean["calculated_host_listings_count"] = safe_numeric(
    airbnb_raw.get("calculated_host_listings_count", np.nan)
)

airbnb_clean["availability_365"] = safe_numeric(
    airbnb_raw.get("availability_365", np.nan)
)

airbnb_clean["number_of_reviews_ltm"] = safe_numeric(
    airbnb_raw.get("number_of_reviews_ltm", np.nan)
)

airbnb_clean["last_review"] = pd.to_datetime(
    airbnb_raw.get("last_review", pd.Series([np.nan] * len(airbnb_raw))),
    errors="coerce"
)

# Coordinates
airbnb_clean["latitude"] = safe_numeric(airbnb_raw["latitude"])
airbnb_clean["longitude"] = safe_numeric(airbnb_raw["longitude"])

## Coordinate check:
We check the coordinates are in range of what barcelona extends

In [111]:
VALID_LAT_MIN = 40.0
VALID_LAT_MAX = 43.5
VALID_LON_MIN = 0.0
VALID_LON_MAX = 3.5

airbnb_clean["has_valid_coordinates"] = (
    airbnb_clean["latitude"].between(VALID_LAT_MIN, VALID_LAT_MAX)
    & airbnb_clean["longitude"].between(VALID_LON_MIN, VALID_LON_MAX)
)

print("Valid Airbnb coordinate rate:", airbnb_clean["has_valid_coordinates"].mean())

display(
    airbnb_clean["has_valid_coordinates"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Valid Airbnb coordinate rate: 1.0


,count
has_valid_coordinates,
True,19410


## Text feature creation:
We use the text to create the variables we mentioned

In [112]:
airbnb_text_features = compute_text_features(airbnb_clean["text"])

for column in airbnb_text_features.columns:
    airbnb_clean[column] = airbnb_text_features[column]

display(
    airbnb_clean[
        [
            "text",
            "text_length",
            "word_count",
            "uppercase_ratio",
            "digit_count",
            "exclamation_count",
        ]
    ].head()
)

,text,text_length,word_count,uppercase_ratio,digit_count,exclamation_count
0,Huge flat for 8 people close to Sagrada Familia,47,9,0.078947,1,0
1,"Forum CCIB DeLuxe, Spacious, Large Balcony, relax",49,7,0.250000,0,0
2,Sagrada Familia area - Còrsega 1,32,6,0.120000,1,0
3,Stylish Top Floor Apartment - Ramblas Plaza Real,48,8,0.175000,0,0
4,VIDRE HOME PLAZA REAL on LAS RAMBLAS,36,7,0.933333,0,0


## Quality checks:
Now we check and create flags for possible mistakes that would not be catched by the nan audit

In [113]:
airbnb_clean["has_invalid_price"] = (
    airbnb_clean["price"].isna()
    | (airbnb_clean["price"] <= 0)
)

airbnb_clean["empty_text"] = (
    airbnb_clean["text"]
    .fillna("")
    .str.len() == 0
)

quality_flag_summary = (
    airbnb_clean[
        [
            "has_invalid_price",
            "empty_text",
            "has_valid_coordinates",
        ]
    ]
    .mean()
    .to_frame("rate")
)

display(quality_flag_summary)

,rate
has_invalid_price,0.212983
empty_text,0.000000
has_valid_coordinates,1.000000


## License Treatement:
We create a different table for the license and our listing key

In [114]:
if "license" not in airbnb_raw.columns:
    raise ValueError("Airbnb raw dataset must contain a 'license' column.")

airbnb_license_reference = pd.DataFrame({
    "listing_key": airbnb_clean["listing_key"],
    "license_raw": airbnb_raw["license"],
})

airbnb_license_reference["license_raw"] = (
    airbnb_license_reference["license_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Airbnb license reference shape:", airbnb_license_reference.shape)
display(airbnb_license_reference.head())

Airbnb license reference shape: (19410, 2)


,listing_key,license_raw
0,airbnb_000000,ESFCTU000008058000039706000000000000000HUTB-002062349
1,airbnb_000001,ESFCTU000008106000547162000000000000000000HUTB0050572
2,airbnb_000002,HUTB-001722
3,airbnb_000003,Exempt
4,airbnb_000004,ESFCTU000008119000093652000000000000000HUTB-001506712


In [115]:
#Here we are just reordering for later
missing_airbnb_silver_columns = [
    column for column in AIRBNB_SILVER_COLUMNS
    if column not in airbnb_clean.columns
]

if missing_airbnb_silver_columns:
    raise ValueError(f"Missing Airbnb Silver columns: {missing_airbnb_silver_columns}")

airbnb_clean = airbnb_clean[AIRBNB_SILVER_COLUMNS]

print("Airbnb Silver shape:", airbnb_clean.shape)
display(airbnb_clean.head())

Airbnb Silver shape: (19410, 25)


,listing_key,text,neighbourhood_group,neighbourhood,municipality,latitude,longitude,has_valid_coordinates,property_type,minimum_nights,availability_365,price,log_price,has_invalid_price,number_of_reviews,last_review,reviews_per_month,number_of_reviews_ltm,calculated_host_listings_count,text_length,word_count,uppercase_ratio,digit_count,exclamation_count,empty_text
0,airbnb_000000,Huge flat for 8 people close to Sagrada Familia,Eixample,la Sagrada Família,la Sagrada Família,41.405560,2.17262,True,Entire home/apt,1,80,210.0,5.347108,False,51,2025-07-31,0.34,7,26,47,9,0.078947,1,0,False
1,airbnb_000001,"Forum CCIB DeLuxe, Spacious, Large Balcony, relax",Sant Martí,el Besòs i el Maresme,el Besòs i el Maresme,41.412432,2.21975,True,Entire home/apt,3,289,285.0,5.652489,False,91,2025-09-08,0.52,12,1,49,7,0.250000,0,0,False
2,airbnb_000002,Sagrada Familia area - Còrsega 1,Gràcia,el Camp d'en Grassot i Gràcia Nova,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,True,Entire home/apt,1,64,170.0,5.135798,False,152,2025-08-08,0.88,23,2,32,6,0.120000,1,0,False
3,airbnb_000003,Stylish Top Floor Apartment - Ramblas Plaza Real,Ciutat Vella,el Barri Gòtic,el Barri Gòtic,41.380620,2.17517,True,Entire home/apt,31,333,110.0,4.700480,False,25,2024-11-05,0.14,5,3,48,8,0.175000,0,0,False
4,airbnb_000004,VIDRE HOME PLAZA REAL on LAS RAMBLAS,Ciutat Vella,el Barri Gòtic,el Barri Gòtic,41.379780,2.17623,True,Entire home/apt,5,335,333.0,5.808142,False,271,2025-08-19,1.49,23,3,36,7,0.933333,0,0,False


## Nan information:
This audit will be usefull for later decision making in the gold dataset

In [116]:
airbnb_missing = (
    airbnb_clean
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

display(airbnb_missing)

,missing_rate
last_review,0.257032
reviews_per_month,0.257032
log_price,0.212983
price,0.212983
municipality,0.000000
text,0.000000
neighbourhood_group,0.000000
neighbourhood,0.000000
listing_key,0.000000
property_type,0.000000


# License dataset treatment

In [117]:
LICENSE_COLUMNS = [
    "NOM_DISTRICTE",
    "NOM_BARRI",
    "NUMERO_PLACES",
    "LONGITUD_X",
    "LATITUD_Y",
]

In [118]:
missing_license_columns = [
    column for column in LICENSE_COLUMNS
    if column not in licenses_raw.columns
]

if missing_license_columns:
    raise ValueError(f"Missing license columns: {missing_license_columns}")

print("All required license columns found.")

All required license columns found.


## Cleaning of the license official dataset:
Now we clean the license dataset: rename the fields, convert types and removing invalid rows and duplicates

In [119]:
licenses_clean = licenses_raw[LICENSE_COLUMNS].copy()

licenses_clean = licenses_clean.rename(columns={
    "NOM_DISTRICTE": "license_district",
    "NOM_BARRI": "license_neighborhood",
    "NUMERO_PLACES": "license_places",
    "LONGITUD_X": "license_x",
    "LATITUD_Y": "license_y",
})

# Numeric conversion robust to comma decimals
for column in ["license_x", "license_y", "license_places"]:
    licenses_clean[column] = (
        licenses_clean[column]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )
    licenses_clean[column] = pd.to_numeric(licenses_clean[column], errors="coerce")

for column in [
    "license_district",
    "license_neighborhood",
]:
    licenses_clean[column] = (
        licenses_clean[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# Keep only rows with usable original coordinates
licenses_clean = licenses_clean.dropna(subset=["license_x", "license_y"]).copy()
licenses_clean = licenses_clean.drop_duplicates().reset_index(drop=True)

print("Licenses Silver initial shape:", licenses_clean.shape)
display(licenses_clean.head())
display(licenses_clean[["license_x", "license_y"]].describe())

Licenses Silver initial shape: (6090, 5)


,license_district,license_neighborhood,license_places,license_x,license_y
0,CIUTAT VELLA,el Raval,3.0,2.170172,41.378445
1,CIUTAT VELLA,"Sant Pere, Santa Caterina i la Ribera",3.0,2.182004,41.382884
2,CIUTAT VELLA,el Barri Gòtic,3.0,2.173086,41.382224
3,CIUTAT VELLA,"Sant Pere, Santa Caterina i la Ribera",9.0,2.183305,41.383685
4,CIUTAT VELLA,el Raval,8.0,2.167911,41.385020


,license_x,license_y
count,6090.000000,6090.000000
mean,2.165470,41.393542
std,0.017857,0.013241
min,2.093280,41.352592
25%,2.155390,41.382730
50%,2.164629,41.393599
75%,2.175293,41.402943
max,2.219091,41.452785


## Coordinate harmonisation:
Since both datasets use different coordinate variables, so we need to make sure they are in the same format, or transform them. Our standart will be WGS84 

In [120]:
def looks_like_wgs84(lon_series, lat_series):
    """
    Heuristic check for longitude/latitude coordinates in degrees.
    """
    lon = lon_series.dropna()
    lat = lat_series.dropna()

    if len(lon) == 0 or len(lat) == 0:
        return False

    lon_ok = lon.between(-180, 180).mean() > 0.95
    lat_ok = lat.between(-90, 90).mean() > 0.95

    return bool(lon_ok and lat_ok)


def convert_projected_to_wgs84(
    df,
    x_col="license_x",
    y_col="license_y",
    source_crs="EPSG:25831"
):
    """
    Convert projected coordinates to WGS84 lon/lat using pyproj.

    EPSG:25831 corresponds to ETRS89 / UTM zone 31N,
    commonly used for projected coordinates in Catalonia.
    """
    transformer = Transformer.from_crs(
        source_crs,
        "EPSG:4326",
        always_xy=True
    )

    lon, lat = transformer.transform(
        df[x_col].values,
        df[y_col].values
    )

    df = df.copy()
    df["license_longitude"] = lon
    df["license_latitude"] = lat
    df["license_source_crs"] = source_crs

    return df


if looks_like_wgs84(licenses_clean["license_x"], licenses_clean["license_y"]):
    licenses_clean["license_longitude"] = licenses_clean["license_x"]
    licenses_clean["license_latitude"] = licenses_clean["license_y"]
    licenses_clean["license_source_crs"] = "EPSG:4326"
else:
    licenses_clean = convert_projected_to_wgs84(
        licenses_clean,
        x_col="license_x",
        y_col="license_y",
        source_crs="EPSG:25831"
    )

licenses_clean["license_longitude"] = pd.to_numeric(
    licenses_clean["license_longitude"],
    errors="coerce"
)

licenses_clean["license_latitude"] = pd.to_numeric(
    licenses_clean["license_latitude"],
    errors="coerce"
)

print("License coordinate CRS detected/used:")
display(licenses_clean["license_source_crs"].value_counts())

display(
    licenses_clean[
        [
            "license_x",
            "license_y",
            "license_longitude",
            "license_latitude",
            "license_source_crs",
        ]
    ].head()
)

License coordinate CRS detected/used:


license_source_crs
EPSG:4326    6090
Name: count, dtype: int64

,license_x,license_y,license_longitude,license_latitude,license_source_crs
0,2.170172,41.378445,2.170172,41.378445,EPSG:4326
1,2.182004,41.382884,2.182004,41.382884,EPSG:4326
2,2.173086,41.382224,2.173086,41.382224,EPSG:4326
3,2.183305,41.383685,2.183305,41.383685,EPSG:4326
4,2.167911,41.385020,2.167911,41.385020,EPSG:4326


In [121]:
licenses_clean["has_valid_coordinates"] = (
    licenses_clean["license_latitude"].between(VALID_LAT_MIN, VALID_LAT_MAX)
    & licenses_clean["license_longitude"].between(VALID_LON_MIN, VALID_LON_MAX)
)

print("Valid license coordinate rate:", licenses_clean["has_valid_coordinates"].mean())

display(
    licenses_clean["has_valid_coordinates"]
    .value_counts(dropna=False)
    .to_frame("count")
)

# Keep only valid coordinates for the Silver license dataset
licenses_clean = (
    licenses_clean[licenses_clean["has_valid_coordinates"]]
    .copy()
    .reset_index(drop=True)
)

print("Licenses Silver shape after coordinate validation:", licenses_clean.shape)

Valid license coordinate rate: 1.0


,count
has_valid_coordinates,
True,6090


Licenses Silver shape after coordinate validation: (6090, 9)


## Silver nan audit

In [122]:
licenses_missing = (
    licenses_clean
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

display(licenses_missing)

,missing_rate
license_places,0.008539
license_district,0.000000
license_neighborhood,0.000000
license_x,0.000000
license_y,0.000000
license_longitude,0.000000
license_latitude,0.000000
license_source_crs,0.000000
has_valid_coordinates,0.000000


In [123]:
def build_quality_summary(df, dataset_name):
    """
    Build a compact data quality summary for a dataframe.
    """
    return pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "dtype": [str(df[column].dtype) for column in df.columns],
        "missing_rate": [df[column].isna().mean() for column in df.columns],
        "non_null_count": [df[column].notna().sum() for column in df.columns],
        "unique_count": [df[column].nunique(dropna=True) for column in df.columns],
    })


silver_quality_summary = pd.concat(
    [
        build_quality_summary(airbnb_clean, "airbnb_clean"),
        build_quality_summary(licenses_clean, "licenses_clean"),
        build_quality_summary(airbnb_license_reference, "airbnb_license_reference"),
    ],
    ignore_index=True
)

display(
    silver_quality_summary
    .sort_values(["dataset", "missing_rate"], ascending=[True, False])
)

,dataset,column,dtype,missing_rate,non_null_count,unique_count
15,airbnb_clean,last_review,datetime64[us],0.257032,14421,1837
16,airbnb_clean,reviews_per_month,float64,0.257032,14421,835
11,airbnb_clean,price,float64,0.212983,15276,772
12,airbnb_clean,log_price,float64,0.212983,15276,772
0,airbnb_clean,listing_key,str,0.000000,19410,19410
1,airbnb_clean,text,str,0.000000,19410,18531
2,airbnb_clean,neighbourhood_group,str,0.000000,19410,10
3,airbnb_clean,neighbourhood,str,0.000000,19410,71
4,airbnb_clean,municipality,str,0.000000,19410,71
5,airbnb_clean,latitude,float64,0.000000,19410,10589


In [124]:
airbnb_clean.to_parquet(AIRBNB_SILVER_PATH, index=False)
licenses_clean.to_parquet(LICENSES_SILVER_PATH, index=False)
airbnb_license_reference.to_parquet(AIRBNB_LICENSE_REFERENCE_PATH, index=False)

silver_quality_summary.to_csv(
    SILVER_QUALITY_SUMMARY_PATH,
    index=False
)

print("Saved:", AIRBNB_SILVER_PATH)
print("Saved:", LICENSES_SILVER_PATH)
print("Saved:", AIRBNB_LICENSE_REFERENCE_PATH)
print("Saved:", SILVER_QUALITY_SUMMARY_PATH)

Saved: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\data\silver\airbnb_clean.parquet
Saved: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\data\silver\licenses_clean.parquet
Saved: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\data\silver\airbnb_license_reference.parquet
Saved: c:\Users\David\Documents\TFG\Fraudulent_property_listing_detection_tool\datasets\reports\tables\silver_data_quality_summary.csv
